# ASTERIX `.ast` notebook: decoding `.ast` to `.csv`

This notebook is a **structured learning notebook** for ASTERIX binary files.  
It is **not** a full decoder yet. Its goal is to explain the **core message structure** that must be understood before converting `.ast` files into clean tabular data.

## 1. Outer structure of an ASTERIX file

An ASTERIX `.ast` file is usually a **continuous binary stream** made of consecutive ASTERIX message blocks.  
There is no explicit end marker. Each block defines its own size through the `LEN` field.

Each message block begins with:

- **Octet 1** → `CAT` (category)
- **Octets 2–3** → `LEN` (total block length, including `CAT` and `LEN`)
- **Remaining octets** → **record area**

### Message schema

```text
| [CAT][LEN][record area] | [CAT][LEN][record area] | [CAT][LEN][record area] | ...
```

```text
Message block
├── CAT
├── LEN
└── Record area
```

The parser therefore advances block by block using:

```text
read CAT -> read LEN -> extract LEN bytes -> move to next block
```

---

## 2. Internal structure of the record area

The record area contains **one or more records**.  
Each record starts with an **FSPEC** and is followed by the **Data Fields** announced by that FSPEC.

### Records schema

```text
Record
├── FSPEC
├── Data Field
├── Data Field
└── ...
```

```text
ASTRIX block
├── CAT
├── LEN
└── Record area
    ├── Record 1
    │   ├── FSPEC
    │   ├── Data Field
    │   └── ...
    ├── Record 2
    │   ├── FSPEC
    │   ├── Data Field
    │   └── ...
    └── ...
```

Key idea:  
**The FSPEC comes first, and the Data Fields follow in the order defined by the category profile.**

---

## 3. What the FSPEC does

The **FSPEC** (*Field Specification*) is a **presence map**.  
It does not contain the actual values. Its job is to say **which fields are present** in the current record.

Each FSPEC octet contains:

- **7 field presence bits**
- **1 FX bit**

If `FX = 1`, another FSPEC octet follows.  
If `FX = 0`, the FSPEC ends.

### FSPEC schema

```text
| FRN1 | FRN2 | FRN3 | FRN4 | FRN5 | FRN6 | FRN7 | FX | FRN8 | FRN9 | ...
```

```text
FSPEC parsing
├── Read first FSPEC octet
├── Check FX
├── If FX = 1 -> read next FSPEC octet
└── Continue until FX = 0
```

The FSPEC tells us which FRNs are present. FRNs are field positions, not yet semantic field names.
After the FSPEC, the actual Data Fields are transmitted in **increasing FRN order**.

### Example conceptually: 
then the record body is organized as:
```text
          | FRN1 | FRN2 | FRN3 | FRN4 | FRN5 | FRN6 | FRN7 |  FX  |
(FSPEC)   | [1]  | [0]  | [1]  | [0]  | [0]  | [1]  | [0]  |  [0] |

Then based on that FSPEC the fields that the remaining octets of that record will be:
 Data           
Fields    | Field for FRN 1 | Field for FRN 3 | Field for FRN 6 |
```

---

## 4. Data Field to Data Item using UAP tables

These two concepts are different:

- **Data Field** → binary representation transmitted in the record. Ex: PRN1, PRN4, PRN9... 
- **Data Item** → logical information defined by the specification. Ex: I021/010 (Data Source Identification), I021/040 (Time of Day)...

The **UAP** (*User Application Profile*) is the category-specific table that maps Data Fields to Data Items (each CATegori has its own mapping):

### Example conceptually: 

```text
    UAP table
FRN 1 ----> I021/010
FRN 2 ----> I021/040
FRN 3 ----> I021/161
```
---

## 5. Data Items length

Once a Data Item is identified, the decoder must know **how many bytes belong to its Data Field**.

This length can be classified in 5 different types and this is specified in the EUROCONTROL documentation:

### Fixed length
Always the same size.

```text
[2 octets always]
```

### Extended
A primary part may continue into one or more extension parts using `FX`.

```text
[Primary][FX=1][Extension][FX=0]
```

### Explicit length
The field begins with a length indicator.

```text
[LENGTH][data...]
```

### Repetitive
The field starts with `REP`, followed by repeated substructures.

```text
[REP][element][element][element]
```

### Compound
The field contains internal subfields controlled by an internal presence map.

```text
[Primary subfield][Subfield 1][Subfield 3][Subfield 5]
```

### Clarifying schema

```text
Data Field formats
├── Fixed length
├── Extended
├── Explicit length
├── Repetitive
└── Compound
```

#### **IMPORTANT WARNING**

A single ASTERIX message may contain *more than one record*.  
However, after the FSPEC, the record body may contain variable-length fields.
So the decoder cannot safely jump to *Record 2* unless it knows the exact total size of *Record 1*.

---

## 6. Decoding Data Items

Once each data item length is determined, they must be separated in order to extract the information they are transmiting.
To do so, the EUROCONTROL documentations provide how each item must be analyzed in order to decode the information properly.
Each item has its own format so be careful understanding the structure of each item.

### Example conceptually: 
This example shows how a single octet can be split into several subfields.  
Each bit or group of bits has a defined meaning.

```text
 8   7   6   5   4   3   2   1
+---+---+---+---+---+---+---+---+
|    ATP    | ARC | RC|RAB| FX |
+-----------+-----+---+---+----+
   bits 8/6   5/4   3   2    1

ATP = Reserved for future use
    0 -> 24-Bit ICAO address
    1 -> Duplicate address
    2 -> Surface vehicle address
    3 -> Anonymous address
    4-7 -> Reserved for future use
ARC = Unknown
    0 -> 25 ft
    1 -> 100 ft
    2 -> Unknown
    3 -> Invalid
RC  = Range Check passed, CPR Validation pending
    0 -> Default
    1 -> Range Check passed, CPR Validation pending
RAB = Report from target transponder
    0 -> Report from target transponder
    1 -> Report from field monitor (fixed transponder)
FX  = Extension into first extension
    0 -> End of item
    1 -> Extension into first extension
```

---
## FINAL CLARIFYING SCHEMA

```text
ASTRIX `.ast` file
├── Message block
│   ├── CAT (cat021 or cat cat048)
│   ├── LEN
│   └── Record area
│       ├── Record
│       │   ├── FSPEC --> FRNs detected
│       │   │                UAP tables lookup                
│       │   ├── Data Fields (FRNx) -->  Data Item (I021/020 - Emitter Category)
│       │   ├── Data Fields (FRNy) -->  Data Item (I021/145 - Flight Level)
│       │   └── Data Fields (FRNz) -->  Data Item (I021/230 - Roll Angle)
│       └── ...
└── Next message block
```

This is the correct foundation before decoding values and exporting them into `.csv`.

---
---


## Block 1 — Load the binary file and validate the first ASTERIX header

This first block loads the binary stream into memory and performs a **minimal header sanity check**.

### Theory behind this block

At the file level, ASTERIX is not a text format and not a line-based format. It is a **binary stream**. The parser must therefore read bytes directly.

The first three octets already tell us a lot:

- `CAT` identifies **which category specification** must be used later.
- `LEN` tells us **how many bytes belong to the current block**.
- Because `LEN` includes the full message size, it is the value that lets us move safely from one message to the next.

At this stage we are **not decoding the payload yet**. We are only confirming that the binary file is readable and that the first header looks like a plausible ASTERIX message header.

### Data structure produced by this block

- `raw_data` → the complete binary file as `bytes`
- `raw_view` → a `memoryview` of the same bytes for efficient slicing later

### Expected output

A short summary showing:

- the file path used
- total number of bytes loaded
- the first `CAT`
- the first `LEN`
- a small hexadecimal preview of the stream


In [5]:

from pathlib import Path
import pandas as pd


def first_existing_path(candidates):
    """Return the first existing path from a list of candidates."""
    for candidate in map(Path, candidates):
        if candidate.exists():
            return candidate
    return None


# Update these candidates to match your local project structure.
BINARY_CANDIDATES = [
    Path("raw_data/datos_asterix_radar.ast"),
]

BINARY_PATH = first_existing_path(BINARY_CANDIDATES)
if BINARY_PATH is None:
    raise FileNotFoundError(
        "No ASTERIX binary file was found. "
        "Update BINARY_CANDIDATES or set BINARY_PATH directly."
    )

# Load the full binary stream once.
raw_data = BINARY_PATH.read_bytes()
raw_view = memoryview(raw_data)

# Extract the first ASTERIX header if enough bytes are available.
if len(raw_data) < 3:
    raise ValueError("The file is too short to contain a valid ASTERIX header.")

first_cat = raw_data[0]
first_len = int.from_bytes(raw_data[1:3], byteorder="big")
first_bytes_hex = raw_data[:24].hex(" ").upper()

file_summary = pd.DataFrame(
    [
        {
            "file": str(BINARY_PATH),
            "size_bytes": len(raw_data),
            "first_cat": first_cat,
            "first_len": first_len,
        }
    ]
)

print(f"Loaded binary file: {BINARY_PATH}")
print(f"Total bytes: {len(raw_data):,}")
print(f"First header -> CAT={first_cat} | LEN={first_len}")
print(f"First 24 bytes (hex): {first_bytes_hex}")


Loaded binary file: raw_data\datos_asterix_radar.ast
Total bytes: 30,060,083
First header -> CAT=48 | LEN=59
First 24 bytes (hex): 30 00 3B FF F7 02 14 81 1C 20 9C A0 C6 2C B2 1E 04 0D 03 69 E0 2F 05 A9


In [9]:
# --- Execution configuration ---
SUPPORTED_CATEGORIES = {21, 48}

# Choose one mode: [21], [48], or [21, 48]
TARGET_CATEGORIES = [21, 48]

INVALID_CATEGORIES = [cat for cat in TARGET_CATEGORIES if cat not in SUPPORTED_CATEGORIES]
if INVALID_CATEGORIES:
    raise ValueError(
        f"Unsupported categories in TARGET_CATEGORIES: {INVALID_CATEGORIES}. "
        f"Supported values are: {sorted(SUPPORTED_CATEGORIES)}"
    )

# Normalize and keep user order without duplicates
TARGET_CATEGORIES = list(dict.fromkeys(int(cat) for cat in TARGET_CATEGORIES))
CSV_OUTPUT_TEMPLATE = "message_data_CAT{cat:03d}.csv"

print(f"TARGET_CATEGORIES: {TARGET_CATEGORIES}")
print(f"CSV_OUTPUT_TEMPLATE: {CSV_OUTPUT_TEMPLATE}")

TARGET_CATEGORIES: [21, 48]
CSV_OUTPUT_TEMPLATE: message_data_CAT{cat:03d}.csv


## Execution modes for category-independent CSV output

Use `TARGET_CATEGORIES` in the configuration cell to choose the export mode:

- `[48]` -> decodes and exports only `message_data_CAT048.csv`
- `[21]` -> decodes and exports only `message_data_CAT021.csv`
- `[21, 48]` -> decodes both and exports two separate CSV files

This notebook now keeps shared parsing steps, but UAP loading, item instancing, decoding, and CSV export run only for the selected categories.


## Block 2 — Split the binary stream into ASTERIX message blocks

This block walks through the file and separates it into **complete ASTERIX messages** using the header rule:

- read `CAT` from octet 1
- read `LEN` from octets 2-3
- slice exactly `LEN` bytes
- jump to the next block

### Theory behind this block

An ASTERIX `.ast` file does **not** need separators between messages. The parser advances using the declared block length. This is why `LEN` is so important: it is the only reliable way to know where the next message starts.

`LEN` is the **total length of the message block**, which means:

```text
message_bytes = [CAT][LEN][record area]
```

So, for every message:

- `message_bytes[:3]` is the message block header
- `message_bytes[3:]` is the record area


### Data structures produced by this block

- `messages` → list of dictionaries, one dictionary per message
- `messages_df` → a compact pandas table summarizing the message boundaries

### Expected output

A concise summary with:

- total number of ASTERIX messages found
- count of messages per category
- a small preview table with offsets, categories, and lengths


In [6]:

def split_asterix_messages(raw_bytes: bytes):
    """Split a binary ASTERIX stream into message dictionaries."""
    stream = memoryview(raw_bytes)
    total_size = len(stream)
    messages = []
    offset = 0
    message_id = 1

    while offset < total_size:
        # Stop cleanly if fewer than 3 bytes remain.
        if offset + 3 > total_size:
            break

        cat = stream[offset]
        length = int.from_bytes(stream[offset + 1: offset + 3], byteorder="big")

        # Reject impossible message lengths.
        if length < 3:
            raise ValueError(f"Invalid LEN={length} at offset {offset}.")
        if offset + length > total_size:
            raise ValueError(
                f"Message at offset {offset} declares LEN={length}, "
                f"which exceeds the file size."
            )

        block = bytes(stream[offset: offset + length])
        record_area = block[3:]

        messages.append(
            {
                "message_id": message_id,
                "offset": offset,
                "cat": cat,
                "length": length,
                "payload_len": length - 3,
                "message_bytes": block,
                "record_area": record_area,
            }
        )

        offset += length
        message_id += 1

    return messages


messages = split_asterix_messages(raw_data)

messages_df = pd.DataFrame.from_records(
    [
        {
            "message_id": msg["message_id"],
            "offset": msg["offset"],
            "cat": msg["cat"],
            "length": msg["length"],
            "payload_len": msg["payload_len"],
        }
        for msg in messages
    ]
)

cat_counts = (
    messages_df["cat"]
    .value_counts()
    .sort_index()
    .rename_axis("CAT")
    .reset_index(name="messages")
)

print(f"ASTERIX messages found: {len(messages_df)}")
print("Messages per CAT:")
print(cat_counts.to_string(index=False))
print()
print("Message table preview:")
print(messages_df.head(8).to_string(index=False))


ASTERIX messages found: 503698
Messages per CAT:
 CAT  messages
  48    503698

Message table preview:
 message_id  offset  cat  length  payload_len
          1       0   48      59           56
          2      59   48      63           60
          3     122   48      71           68
          4     193   48      59           56
          5     252   48      57           54
          6     309   48      67           64
          7     376   48      65           62
          8     441   48      59           56



## Block 3 — Parse the FSPEC of the first record and list the announced FRNs

This block extracts the **FSPEC of the first record only** from each message.

### Theory behind this block

Inside each record, the FSPEC is the structure that announces **which fields are present**. It is not a value field; it is a **presence map**.

Each FSPEC octet works like this:

- the first 7 bits announce field presence (`FRN1`, `FRN2`, ..., `FRN7`, then `FRN8`, ... in later octets)
- the last bit is **FX**
- if `FX = 1`, another FSPEC octet follows
- if `FX = 0`, the FSPEC ends

So the FSPEC can be one octet long or several octets long.

### Why only the first record?

A message may contain multiple records, but after the FSPEC the record body contains items with different internal formats:

- **fixed length** → the number of octets is known immediately
- **variable / extended length** → an internal `FX` may extend the field
- **repetitive** → a repetition counter (`REP`) defines the size
- **compound** → a primary subfield announces which subfields follow

Because of that, the start of the next record cannot be found from the FSPEC alone. A correct parser would need to decode **all Data Fields of record #1** before it could safely locate **record #2**.

### Data structure produced by this block

- `fspec_df` → one row per message with:
  - message id
  - category
  - FSPEC in hex
  - FSPEC length in octets
  - list of FRNs detected in the first record

### Expected output

A concise table such as:

```text
Message 1 | CAT=21 | FSPEC=A0 | FRNs=FRN1, FRN3
```

This confirms that the message boundaries are correct and that the first-record presence map has been interpreted properly.


In [7]:

def parse_first_record_fspec(record_area: bytes):
    """Parse the FSPEC found at the start of a record area.

    Returns
    -------
    fspec_octets : list[int]
        Raw FSPEC octets.
    frns : list[int]
        FRNs announced by the FSPEC, in increasing order.
    """
    if not record_area:
        return [], []

    fspec_octets = []
    frns = []
    frn_base = 1
    cursor = 0

    while cursor < len(record_area):
        octet = record_area[cursor]
        fspec_octets.append(octet)

        # Bits 8..2 map to seven consecutive FRNs.
        current_frns = list(range(frn_base, frn_base + 7))
        current_bits = [((octet >> bit_shift) & 1) for bit_shift in range(7, 0, -1)]
        frns.extend(frn for frn, is_present in zip(current_frns, current_bits) if is_present)

        # Bit 1 is FX.
        fx = octet & 0b1
        cursor += 1
        frn_base += 7

        if fx == 0:
            break

    return fspec_octets, frns


fspec_rows = []
for msg in messages:
    fspec_octets, frns = parse_first_record_fspec(msg["record_area"])
    fspec_rows.append(
        {
            "message_id": msg["message_id"],
            "cat": msg["cat"],
            "fspec_hex": " ".join(f"{octet:02X}" for octet in fspec_octets),
            "fspec_len": len(fspec_octets),
            "frns": frns,
            "frn_text": ", ".join(f"FRN{frn}" for frn in frns) if frns else "(none)",
        }
    )

fspec_df = pd.DataFrame.from_records(fspec_rows)

print(f"First-record FSPEC parsed for {len(fspec_df)} messages.")
print(
    fspec_df[["message_id", "cat", "fspec_hex", "frn_text"]]
    .head(10)
    .to_string(index=False)
)


First-record FSPEC parsed for 503698 messages.
 message_id  cat fspec_hex                                                                                frn_text
          1   48  FF F7 02 FRN1, FRN2, FRN3, FRN4, FRN5, FRN6, FRN7, FRN8, FRN9, FRN10, FRN11, FRN13, FRN14, FRN21
          2   48  FF F7 02 FRN1, FRN2, FRN3, FRN4, FRN5, FRN6, FRN7, FRN8, FRN9, FRN10, FRN11, FRN13, FRN14, FRN21
          3   48  FF F7 02 FRN1, FRN2, FRN3, FRN4, FRN5, FRN6, FRN7, FRN8, FRN9, FRN10, FRN11, FRN13, FRN14, FRN21
          4   48  FF F7 02 FRN1, FRN2, FRN3, FRN4, FRN5, FRN6, FRN7, FRN8, FRN9, FRN10, FRN11, FRN13, FRN14, FRN21
          5   48  FB F7 02       FRN1, FRN2, FRN3, FRN4, FRN5, FRN7, FRN8, FRN9, FRN10, FRN11, FRN13, FRN14, FRN21
          6   48  FF F7 02 FRN1, FRN2, FRN3, FRN4, FRN5, FRN6, FRN7, FRN8, FRN9, FRN10, FRN11, FRN13, FRN14, FRN21
          7   48  FB F7 02       FRN1, FRN2, FRN3, FRN4, FRN5, FRN7, FRN8, FRN9, FRN10, FRN11, FRN13, FRN14, FRN21
          8   48  FF F7 02 FRN1, 


## Block 4 — Map FRNs to Data Items using the category UAP tables

This block translates the FSPEC result into **human-readable Data Items**.

### Theory behind this block

The FSPEC tells us **which FRNs are present**, but it does **not** tell us directly which semantic field each FRN represents. That mapping is defined by the **User Application Profile (UAP)** of the category.

This is why the category matters:

- `FRN 1` in **CAT021** maps to one Data Item
- `FRN 1` in **CAT048** maps to another Data Item

So the real meaning of an FRN is always:

```text
(CAT, FRN) -> Data Item
```

### Why the UAP table is the correct place to look

The UAP is the category-specific table that answers questions such as:

- Which Data Item corresponds to `FRN 1`?
- Which items are fixed length?
- Which items are variable, repetitive, or compound?
- Which FRNs are reserved for `RE` and `SP`?

The **FSPEC only announces presence**. The **UAP gives meaning**.

### Pandas strategy used here

To keep the workflow clean and scalable, this block:

1. loads the CAT021 and CAT048 UAP tables once
2. normalizes them into a single lookup table `uap_df`
3. explodes the FRN lists from `fspec_df`
4. performs a pandas `merge` on `(CAT, FRN)`

This creates a tidy table where each row represents one detected field in one message.

### Important limitation

This block still maps only the **first record** of each message, because that is what the FSPEC block extracted. It does **not** yet decode field values and does **not** yet walk into later records.

### Data structures produced by this block

- `uap_df` → normalized UAP lookup table
- `data_items_df` → one row per detected `(message, CAT, FRN, Data Item)` pair
- `message_items_df` → one summary row per message

### Expected output

A compact per-message summary such as:

```text
Message 1 | CAT=21 | I021/010, I021/161
```

This is the final proof that the notebook correctly connected:

- message slicing
- first-record FSPEC parsing
- category-specific UAP mapping


In [11]:

def read_uap_source(path: Path, sheet_name=None) -> pd.DataFrame:
    """Read either a CSV or an Excel UAP source."""
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=sheet_name)
    raise ValueError(f"Unsupported UAP file format: {path}")


def normalize_uap_table(df: pd.DataFrame, cat: int) -> pd.DataFrame:
    """Normalize a UAP table to a clean CAT/FRN/Data Item schema."""
    normalized = df.rename(
        columns={
            "Information": "Description",
            "Length": "Length in Octets",
        }
    ).copy()

    normalized["FRN_text"] = normalized["FRN"].astype(str).str.strip().str.upper()
    normalized = normalized[~(normalized["FRN_text"].isin(["FX", "-", "N.A.", "NAN"]) | normalized["Data Item"].isin(["-"]))]
    normalized = normalized[normalized["FRN_text"].str.fullmatch(r"\d+")]

    normalized["CAT"] = cat
    normalized["FRN"] = normalized["FRN_text"].astype(int)
    normalized["Data Item"] = normalized["Data Item"].fillna("").astype(str).str.strip()
    normalized["Description"] = normalized["Description"].fillna("").astype(str).str.strip()
    normalized["Length in Octets"] = (
        normalized["Length in Octets"].fillna("").astype(str).str.strip()
    )

    return normalized[["CAT", "FRN", "Data Item", "Description", "Length in Octets"]]


def normalize_builtin_uap_table(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize internal Python UAP tables to CAT/FRN/Data Item schema."""
    normalized = df.rename(
        columns={
            "cat": "CAT",
            "frn": "FRN",
            "item_id": "Data Item",
            "item_name": "Description",
            "length_str": "Length in Octets",
        }
    ).copy()
    normalized["CAT"] = normalized["CAT"].astype(int)
    normalized["FRN"] = normalized["FRN"].astype(int)
    normalized["Data Item"] = normalized["Data Item"].fillna("").astype(str).str.strip()
    normalized["Description"] = normalized["Description"].fillna("").astype(str).str.strip()
    normalized["Length in Octets"] = normalized["Length in Octets"].fillna("").astype(str).str.strip()
    return normalized[["CAT", "FRN", "Data Item", "Description", "Length in Octets"]]


def load_uap_table(cat: int, candidates, sheet_name=None) -> pd.DataFrame:
    """Load the first available UAP source for one category."""
    source_path = first_existing_path(candidates)
    if source_path is None:
        raise FileNotFoundError(
            f"No UAP file found for CAT{cat:03d}. Update the candidate paths."
        )
    raw_table = read_uap_source(source_path, sheet_name=sheet_name)
    return normalize_uap_table(raw_table, cat=cat)


def load_builtin_uap_table(cat: int) -> pd.DataFrame:
    """Load fallback UAP table from internal Python definitions."""
    from asterix_decoder.data_tables.uap_tables import uap021_df, uap048_df

    builtin_by_cat = {
        21: uap021_df,
        48: uap048_df,
    }
    table = builtin_by_cat.get(cat)
    if table is None:
        raise KeyError(f"No builtin UAP table available for CAT{cat:03d}.")
    return normalize_builtin_uap_table(table)


uap_sources = {
    21: {
        "candidates": [
            Path("asterix_decoder/uap_tables/CAT021.csv"),
        ],
        "sheet_name": "CAT021_UAP",
    },
    48: {
        "candidates": [
            Path("asterix_decoder/uap_tables/CAT048.csv"),
        ],
        "sheet_name": "CAT048_UAP",
    },
}

selected_messages_df = messages_df[messages_df["cat"].isin(TARGET_CATEGORIES)].copy()
selected_fspec_df = fspec_df[fspec_df["cat"].isin(TARGET_CATEGORIES)].copy()

loaded_uap_tables = []
uap_source_used = {}
for cat in TARGET_CATEGORIES:
    cfg = uap_sources.get(cat)
    if cfg is None:
        raise KeyError(f"UAP source not configured for CAT{cat:03d}.")
    try:
        loaded_uap_tables.append(load_uap_table(cat, cfg["candidates"], cfg["sheet_name"]))
        uap_source_used[cat] = "file"
    except FileNotFoundError:
        loaded_uap_tables.append(load_builtin_uap_table(cat))
        uap_source_used[cat] = "builtin"

uap_df = pd.concat(loaded_uap_tables, ignore_index=True)

message_frn_df = (
    selected_fspec_df.rename(columns={"cat": "CAT"})[["message_id", "CAT", "frns"]]
    .explode("frns")
    .rename(columns={"frns": "FRN"})
    .dropna(subset=["FRN"])
    .copy()
)
message_frn_df["FRN"] = message_frn_df["FRN"].astype(int)

# Join the detected FRNs with the UAP lookup.
data_items_df = (
    message_frn_df
    .merge(uap_df, on=["CAT", "FRN"], how="left")
    .sort_values(["message_id", "FRN"], kind="stable")
    .reset_index(drop=True)
)

unknown_mask = data_items_df["Data Item"].isna() | data_items_df["Data Item"].eq("")
data_items_df.loc[unknown_mask, "Data Item"] = (
    data_items_df.loc[unknown_mask]
    .apply(lambda row: f"I{int(row['CAT']):03d}/FRN{int(row['FRN'])}", axis=1)
)
data_items_df["Description"] = data_items_df["Description"].fillna("(unknown)")

message_items_df = (
    data_items_df.assign(item_label=data_items_df["Data Item"])
    .groupby(["message_id", "CAT"], as_index=False)
    .agg(
        item_count=("FRN", "size"),
        data_items=("item_label", lambda values: ", ".join(values)),
    )
    .sort_values("message_id", kind="stable")
    .reset_index(drop=True)
)

print(f"Selected messages: {len(selected_messages_df)} / {len(messages_df)}")
print(f"Selected FSPEC rows: {len(selected_fspec_df)} / {len(fspec_df)}")
print(f"UAP sources used: {uap_source_used}")
print(f"UAP rows loaded: {len(uap_df)} | Total Data Items: {len(data_items_df)}")
print(f"Unique Data Items: ({data_items_df['Data Item'].nunique()}) {data_items_df['Data Item'].unique().tolist()}")
print()
print("Per-message Data Item summary:")
print(message_items_df.head(10).to_string(index=False))


Selected messages: 503698 / 503698
Selected FSPEC rows: 503698 / 503698
UAP sources used: {21: 'builtin', 48: 'builtin'}
UAP rows loaded: 72 | Total Data Items: 6888892
Unique Data Items: (17) ['I048/010', 'I048/140', 'I048/020', 'I048/040', 'I048/070', 'I048/090', 'I048/130', 'I048/220', 'I048/240', 'I048/250', 'I048/161', 'I048/200', 'I048/170', 'I048/230', 'I048/030', 'I048/080', 'I048/100']

Per-message Data Item summary:
 message_id  CAT  item_count                                                                                                                                 data_items
          1   48          14 I048/010, I048/140, I048/020, I048/040, I048/070, I048/090, I048/130, I048/220, I048/240, I048/250, I048/161, I048/200, I048/170, I048/230
          2   48          14 I048/010, I048/140, I048/020, I048/040, I048/070, I048/090, I048/130, I048/220, I048/240, I048/250, I048/161, I048/200, I048/170, I048/230
          3   48          14 I048/010, I048/140, I048/020, I048/04

## Block 5 — Load all the item instances dynamically


In [12]:
import inspect
import importlib.util
from pathlib import Path
from asterix_decoder.data_items.data_item import ItemXXX


def build_item_instances(
    uap_table: pd.DataFrame,
    base_folder: Path | str = Path("asterix_decoder/data_items"),
) -> dict[str, object]:
    """Load Data Item classes from disk and instantiate them using the UAP table.

    The category is inferred from the CAT column of uap_table.
    The corresponding folder is <base_folder>/CAT<cat:03d>/ (e.g. CAT048/).
    Items without a concrete class receive an ItemXXX placeholder.

    Parameters
    ----------
    uap_table : pd.DataFrame
        Normalized UAP table with columns: CAT, FRN, Data Item, Description, Length in Octets.
        Must contain rows for exactly one category.
    base_folder : Path | str
        Root folder containing per-category subfolders.

    Returns
    -------
    dict[str, object]
        Mapping from item_id (e.g. 'I048/010') to its instantiated DataItem.
    """
    base_folder = Path(base_folder)
    cats = uap_table["CAT"].unique()
    if len(cats) != 1:
        raise ValueError(
            f"uap_table must contain exactly one CAT value, got {cats.tolist()}. "
            "Filter the table before calling this function."
        )
    cat_int = int(cats[0])
    folder_path = base_folder / f"CAT{cat_int:03d}"

    # --- Phase 1: load concrete classes from the category folder ---
    class_by_item_id: dict[str, type] = {}
    if folder_path.exists():
        for file_path in folder_path.glob("*.py"):
            if file_path.name == "__init__.py":
                continue

            module_name = f"cat{cat_int:03d}_{file_path.stem}"
            spec = importlib.util.spec_from_file_location(module_name, file_path)
            if spec is None or spec.loader is None:
                continue

            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)

            for _, cls in inspect.getmembers(module, inspect.isclass):
                if cls.__module__ != module_name:
                    continue
                get_item_id_fn = getattr(cls, "get_item_id", None)
                if callable(get_item_id_fn):
                    class_by_item_id[get_item_id_fn()] = cls

    # --- Phase 2: instantiate one object per UAP row ---
    instances_by_item_id: dict[str, object] = {}
    for _, row in uap_table.iterrows():
        item_id = str(row["Data Item"]).strip()
        if not item_id:
            continue

        item_name = str(row["Description"]).strip()
        length_str = str(row["Length in Octets"]).strip()
        item_class = class_by_item_id.get(item_id)

        if item_class is None:
            instances_by_item_id[item_id] = ItemXXX(
                item_name=item_name, length_str=length_str, item_id=item_id
            )
            continue

        try:
            instances_by_item_id[item_id] = item_class(
                item_name=item_name, length_str=length_str
            )
        except TypeError:
            instances_by_item_id[item_id] = item_class(item_name, length_str)

    return instances_by_item_id


items_by_cat = {}
for cat in TARGET_CATEGORIES:
    cat_uap = uap_df[uap_df["CAT"] == cat].copy()
    items_by_cat[cat] = build_item_instances(cat_uap)

for cat in TARGET_CATEGORIES:
    cat_items = items_by_cat.get(cat, {})
    print(f"CAT{cat:03d} instances: {len(cat_items)}")
    print(sorted(cat_items.keys()))
    print()


CAT021 instances: 44
['I021/008', 'I021/010', 'I021/015', 'I021/016', 'I021/020', 'I021/040', 'I021/070', 'I021/071', 'I021/072', 'I021/073', 'I021/074', 'I021/075', 'I021/076', 'I021/077', 'I021/080', 'I021/090', 'I021/110', 'I021/130', 'I021/131', 'I021/132', 'I021/140', 'I021/145', 'I021/146', 'I021/148', 'I021/150', 'I021/151', 'I021/152', 'I021/155', 'I021/157', 'I021/160', 'I021/161', 'I021/165', 'I021/170', 'I021/200', 'I021/210', 'I021/220', 'I021/230', 'I021/250', 'I021/260', 'I021/271', 'I021/295', 'I021/400', 'RE', 'SP']

CAT048 instances: 28
['I048/010', 'I048/020', 'I048/030', 'I048/040', 'I048/042', 'I048/050', 'I048/055', 'I048/060', 'I048/065', 'I048/070', 'I048/080', 'I048/090', 'I048/100', 'I048/110', 'I048/120', 'I048/130', 'I048/140', 'I048/161', 'I048/170', 'I048/200', 'I048/210', 'I048/220', 'I048/230', 'I048/240', 'I048/250', 'I048/260', 'RE-Data Item', 'SP-Data Item']



In [13]:
import copy
from asterix_decoder.data_items.error_exceptions import AsterixDecodeError


prototype_maps = items_by_cat
messages_by_id = {msg["message_id"]: msg for msg in messages}
fspec_len_by_message = fspec_df.set_index("message_id")["fspec_len"].to_dict()

selected_message_items_df = message_items_df[message_items_df["CAT"].isin(TARGET_CATEGORIES)].copy()

# 1) Build one cloned item list per message from message_items_df
message_decoding_plan = {}
plan_rows = []

for row in selected_message_items_df.sort_values("message_id", kind="stable").itertuples(index=False):
    message_id = int(row.message_id)
    cat = int(row.CAT)

    item_ids = [item_id.strip() for item_id in str(row.data_items).split(",") if item_id.strip()]
    prototypes = prototype_maps.get(cat, {})

    cloned_items = []
    for item_id in item_ids:
        prototype = prototypes.get(item_id)
        if prototype is None:
            plan_rows.append(
                {
                    "message_id": message_id,
                    "CAT": cat,
                    "item_id": item_id,
                    "status": "missing_prototype",
                }
            )
            continue

        cloned = copy.deepcopy(prototype)
        cloned_items.append(cloned)

        plan_rows.append(
            {
                "message_id": message_id,
                "CAT": cat,
                "item_id": item_id,
                "status": "ready",
            }
        )

    message_decoding_plan[message_id] = cloned_items

message_decoding_plan_df = pd.DataFrame.from_records(plan_rows)


# 2) Decode message-by-message, chaining offsets item-by-item
#    Decoding starts right after the FSPEC octets of each message.
decoded_rows = []
summary_rows = []

for row in selected_message_items_df.sort_values("message_id", kind="stable").itertuples(index=False):
    message_id = int(row.message_id)
    cat = int(row.CAT)

    message_info = messages_by_id.get(message_id)
    if message_info is None:
        summary_rows.append(
            {
                "message_id": message_id,
                "CAT": cat,
                "status": "missing_message",
                "decoded_items": 0,
                "bytes_consumed": 0,
                "record_area_len": 0,
            }
        )
        continue

    record_area = message_info["record_area"]
    fspec_len = int(fspec_len_by_message.get(message_id, 0))
    payload = record_area[fspec_len:]
    items_for_message = message_decoding_plan.get(message_id, [])

    cursor = 0
    decoded_ok = 0
    status = "ok"

    for order_idx, item_instance in enumerate(items_for_message, start=1):
        start_byte = cursor
        start_in_record_area = fspec_len + start_byte
        remaining = payload[start_byte:]

        if not remaining:
            status = "truncated_record_area"
            decoded_rows.append(
                {
                    "message_id": message_id,
                    "CAT": cat,
                    "item_order": order_idx,
                    "item_id": item_instance.item_id,
                    "start_byte": start_byte,
                    "start_in_record_area": start_in_record_area,
                    "consumed_bytes": 0,
                    "next_start_byte": start_byte,
                    "next_start_in_record_area": start_in_record_area,
                    "octets_hex": "",
                    "decoded_data": {},
                    "status": status,
                    "error": "No bytes left in payload",
                }
            )
            break

        try:
            consumed = int(item_instance.decode(remaining))
            next_start = start_byte + consumed
            next_start_in_record_area = fspec_len + next_start

            decoded_rows.append(
                {
                    "message_id": message_id,
                    "CAT": cat,
                    "item_order": order_idx,
                    "item_id": item_instance.item_id,
                    "start_byte": start_byte,
                    "start_in_record_area": start_in_record_area,
                    "consumed_bytes": consumed,
                    "next_start_byte": next_start,
                    "next_start_in_record_area": next_start_in_record_area,
                    "octets_hex": item_instance.octets.hex(" ").upper(),
                    "decoded_data": item_instance.get_data(),
                    "status": "ok",
                    "error": "",
                }
            )

            cursor = next_start
            decoded_ok += 1

        except (AsterixDecodeError, Exception) as exc:
            status = "decode_error"
            decoded_rows.append(
                {
                    "message_id": message_id,
                    "CAT": cat,
                    "item_order": order_idx,
                    "item_id": item_instance.item_id,
                    "start_byte": start_byte,
                    "start_in_record_area": start_in_record_area,
                    "consumed_bytes": 0,
                    "next_start_byte": start_byte,
                    "next_start_in_record_area": start_in_record_area,
                    "octets_hex": "",
                    "decoded_data": {},
                    "status": status,
                    "error": str(exc),
                }
            )
            break

    summary_rows.append(
        {
            "message_id": message_id,
            "CAT": cat,
            "status": status,
            "decoded_items": decoded_ok,
            "planned_items": len(items_for_message),
            "bytes_consumed": cursor,
            "payload_len": len(payload),
            "record_area_len": len(record_area),
            "remaining_payload_bytes": len(payload) - cursor,
        }
    )


decoded_items_df = pd.DataFrame.from_records(decoded_rows)
message_decode_summary_df = pd.DataFrame.from_records(summary_rows)

print(f"Selected messages with decoding plans: {len(message_decoding_plan)}")
if not message_decoding_plan_df.empty:
    print(message_decoding_plan_df["status"].value_counts(dropna=False).to_string())
else:
    print("No decoding plans generated.")
print()
print("Per-message decode summary:")
print(message_decode_summary_df.head(10).to_string(index=False))
print()
print("First decoded items:")
if not decoded_items_df.empty:
    print(
        decoded_items_df[
            [
                "message_id",
                "CAT",
                "item_order",
                "item_id",
                "start_byte",
                "start_in_record_area",
                "consumed_bytes",
                "next_start_byte",
                "status",
            ]
        ]
        .head(20)
        .to_string(index=False)
    )
else:
    print("No decoded items.")

Selected messages with decoding plans: 503698
status
ready    6888892

Per-message decode summary:
 message_id  CAT       status  decoded_items  planned_items  bytes_consumed  payload_len  record_area_len  remaining_payload_bytes
          1   48 decode_error              0             14               0           53               56                       53
          2   48 decode_error              0             14               0           57               60                       57
          3   48 decode_error              0             14               0           65               68                       65
          4   48 decode_error              0             14               0           53               56                       53
          5   48 decode_error              0             13               0           51               54                       51
          6   48 decode_error              0             14               0           61               64          

In [14]:
from collections import defaultdict
from pathlib import Path


# Use only successfully decoded items
ok_items_df = decoded_items_df[decoded_items_df["status"] == "ok"].copy()

message_data_by_cat = {}
output_csv_paths = {}

for cat in TARGET_CATEGORIES:
    cat_ok_items_df = ok_items_df[ok_items_df["CAT"] == cat].copy()

    if cat_ok_items_df.empty:
        print(f"CAT{cat:03d}: no decoded rows available. Skipping CSV export.")
        print()
        continue

    # 1) Collect unique data keys that have at least one non-None value
    all_data_keys = sorted(
        {
            key
            for data in cat_ok_items_df["decoded_data"]
            if isinstance(data, dict)
            for key, value in data.items()
            if value is not None
        }
    )

    # 2) Build one row per message, filling columns with values from all item instances
    rows_by_message = defaultdict(lambda: defaultdict(list))
    all_message_ids = sorted(cat_ok_items_df["message_id"].astype(int).unique().tolist())

    for row in cat_ok_items_df.itertuples(index=False):
        message_id = int(row.message_id)
        data = row.decoded_data if isinstance(row.decoded_data, dict) else {}
        for key in all_data_keys:
            value = data.get(key, None)
            if value is not None:
                rows_by_message[message_id][key].append(value)

    message_rows = []
    for message_id in all_message_ids:
        row_dict = {
            "message_id": message_id,
            "CAT": cat,
        }

        for key in all_data_keys:
            values = rows_by_message[message_id].get(key, [])
            if not values:
                row_dict[key] = None
            elif len(values) == 1:
                row_dict[key] = values[0]
            else:
                # Keep all values found in the same message for the same key
                row_dict[key] = values

        message_rows.append(row_dict)

    message_data_cat_df = pd.DataFrame(message_rows, columns=["message_id", "CAT", *all_data_keys])
    output_csv_path = Path(CSV_OUTPUT_TEMPLATE.format(cat=cat))
    message_data_cat_df.to_csv(output_csv_path, index=False)

    message_data_by_cat[cat] = message_data_cat_df
    output_csv_paths[cat] = output_csv_path.resolve()

    print(f"CAT{cat:03d} decoded rows used: {len(cat_ok_items_df)}")
    print(f"CAT{cat:03d} messages in output: {len(message_data_cat_df)}")
    print(f"CAT{cat:03d} unique non-None keys: {len(all_data_keys)}")
    print(f"CAT{cat:03d} shape: {message_data_cat_df.shape}")
    print(f"CAT{cat:03d} CSV exported to: {output_csv_path.resolve()}")
    print()

    preview_cols = ["message_id", "CAT", *all_data_keys[:12]]
    print(message_data_cat_df[preview_cols].head(10).to_string(index=False))
    print()

if not output_csv_paths:
    print("No CSV files were exported for the selected categories.")

CAT021: no decoded rows available. Skipping CSV export.

CAT048: no decoded rows available. Skipping CSV export.

No CSV files were exported for the selected categories.
